In [ ]:
#1 Data Understanding

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
from pathlib import Path
#read or transform the directory file path into a object so that we can manipulate it easily
data_path = Path("../data/raw/creditcard.csv")
if data_path.is_file():
    #charge the dataset into a pandas DataFrame
    df = pd.read_csv(data_path)
else:
    raise FileNotFoundError("File not found. Please check the path and try again.")


In [4]:
#Discover the dataset
#print(df.shape)
df.head()
#df.tail()
#df.info()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
#2 Data Quality assessment

In [ ]:
#identify the potential dataset issues
#print(df.isna().sum())

#Missing values check
if df.isna().any().any() == True:
    print("There are missing values in the dataset.")
else:
    print("There are no missing values in the dataset.")

#Duplicate values check
if df.duplicated().any() == True:
    print("There are duplicates in the dataset.")
else:
    print("There are no duplicates in the dataset.")

#check for class imbalance
#we use the value_counts() method to count the number of occurrences of each class in the 'Class' column
class_counts = df['Class'].value_counts()
class_counts_normal = class_counts[0]
class_counts_fraud = class_counts[1]
print(f"Number of normal transactions: {class_counts_normal}")
print(f"Number of fraudulent transactions: {class_counts_fraud}")


There are no missing values in the dataset.
There are duplicates in the dataset.
Number of normal transactions: 284315
Number of fraudulent transactions: 492


In [5]:
#count the number of duplicates fraudulent transactions and normal transactions\
# we add keep = False because within one group of duplicates, we can have little groups of duplicates 
# and it considers the first occurence of these groups of duplicates.
essai = df.duplicated(keep = False)

#the fact is that we can use duplicated() method because it conserves the order pf rows.
#Don't forget that the first occurrence of a duplicate is marked as False, while subsequent occurrences are marked as True. Therefore, we can count the number of duplicates for each class by checking the 'Class' column and the 'duplicated' column.

index_fraud = 0
index_normal = 0
for i in range (len(df)):
    if (df.loc[i, 'Class'] == 1):
        if essai[i] == True:
            index_fraud = index_fraud + 1
    if (df.loc[i, 'Class'] == 0):
        if essai[i] == True:
            index_normal = index_normal + 1    
print("Number of duplicate fraudulent transactions:", index_fraud)
print("Number of duplicate normal transactions:", index_normal)

#percentage of duplicate fraudulent transactions and normal transactions
percentage_fraud = (index_fraud / class_counts_fraud) * 100
percentage_normal = (index_normal / class_counts_normal) * 100

print(f"Percentage of duplicate fraudulent transactions : {percentage_fraud: .2f}%")
print(f"Percentage of duplicate normal transactions : {percentage_normal: .2f}%")


Number of duplicate fraudulent transactions: 32
Number of duplicate normal transactions: 1822
Percentage of duplicate fraudulent transactions :  6.50%
Percentage of duplicate normal transactions :  0.64%


In [ ]:
#groupe of duplicates by class
fraudulent_duplicates = df[df.duplicated(keep = False) & (df['Class'] == 1)]
normal_duplicates = df[df.duplicated(keep = False) & (df['Class']== 0)]

#duplicated give us the occurrences of duplicated while being aware that the first occurrence of duplicates
# is marked as False so we can use value_counts() to count the number of occurrences of duplicates for each class.

fraudulent_grouped = fraudulent_duplicates.duplicated().value_counts()
fraudulent_occ = fraudulent_grouped.iloc[1]
normal_grouped = normal_duplicates.duplicated().value_counts()
#print (fraudulent_grouped)
normal_occ = normal_grouped.iloc[1]
print(fraudulent_occ)
print(normal_occ)

13
760


In [ ]:
#we explore the idea that we can compare a row to the entire dataframe and at any groupe of duplicates (which is recognise by 'False')
# --For fraudulent transactions and normal transactions, we define a fonction--

def dist_duplicate_group(tab_ver):

    tab2 = []
    duplicates_flags = tab_ver.duplicated()
    x = 0
    for i in range (len(tab_ver)):
        if (duplicates_flags.iloc[i] == False):
            ligne_test = tab_ver.iloc[i]
            list_test = tab_ver.eq(ligne_test).all(axis = 1)
            for j in range(len(list_test)):
                if(list_test.iloc[j] == True):
                    x = x + 1
            tab2.append(x)
            x = 0    
    return(tab2)

tab3 = dist_duplicate_group(fraudulent_duplicates)
print(f"The number of occurrences per group of duplicates for the fraudulent transactions: {tab3}")

tab4 = dist_duplicate_group(normal_duplicates)
print(f"The number of occurences per group of duplicates for the normal transactions : {tab4}")

The number of occurrences per group of duplicates for the fraudulent transactions: [6, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2]
The number of occurences per group of duplicates for the normal transactions : [2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 2, 2, 4, 4, 4, 2, 2, 4, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 3, 3, 3, 2, 2, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 2, 2, 2, 2, 3, 3, 3, 2, 2, 2, 2, 4, 2, 2, 2, 2, 9, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 2, 2, 5, 5, 5, 5, 5, 2, 2, 2, 2, 3, 3, 3, 2, 2, 3, 3,

In [ ]:
#experimentation cell

tab4_dataf = pd.DataFrame(tab4, columns = ["Values"])
print(tab4_dataf.value_counts())
tab3_dataf = pd.DataFrame(tab3, columns = ["Values"])
print(tab3_dataf.value_counts())

Values
2         600
4          80
3          66
5          10
9           2
18          2
Name: count, dtype: int64
Values
2         11
6          1
4          1
Name: count, dtype: int64


Duplicate Analysis

Given the results obtained previously, we are focusing on duplicates (excluding missing values).

Objective:
Analyze the presence and distribution of duplicate records in the dataset, and assess their potential impact on the subsequent splitting of the data into training, validation, and test sets.

Observations:

The analysis revealed the presence of duplicates, broken down by class as follows:

-- Fraudulent transactions (Class = 1): 32 records belonging to 13 duplicate groups, representing 6.50% of the total number of fraudulent transactions.
-- Normal transactions (Class = 0): 1,822 records belonging to 760 duplicates groups, representing 0.64% of the total number of normal transactions.

An analysis of the occurrences within these duplicate groups revealed:

-- The average number of occurrences per group is approximately 2.4 for both classes.
-- Some duplicate groups contain as many as 18 occurrences (found exclusively within the normal transactions).

Interpretation and Decisions:

The presence of exact duplicates, including identical values for the Time variable, raises the possibility that some transactions may have been recorded multiple times.

Although we cannot establish with certainty whether these repetitions correspond to duplicate records or distinct transactions sharing identical characteristics, keeping all occurrences could lead to an overrepresentation of certain observations.

Furthermore, if identical records are distributed across the training, validation, and test sets, this could result in data leakage and potentially lead to an overestimation of model performance.

Therefore, we propose removing exact duplicates during the Data Preprocessing stage, keeping only one occurrence of each unique record. The original dataset will be preserved.

In [ ]:
#Check the types of datas, inconsistent values
#print(df['Amount'].dtypes)
#check the infini

#for i in range (len(df)):
    



The column 0 type is Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object 

The column 1 type is Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21   

In [65]:
#print((type(df.loc[0, 'Amount'])) in (df['Amount'].dtypes))
typ1 = type(df.loc[0, 'Amount'])
typ2 = df['Amount'].dtypes
print(type(typ2))
print(type(typ1))
print(df.shape)


<class 'numpy.dtypes.Float64DType'>
<class 'type'>
(284807, 31)
